# OTC FX Spot Process Analysis

Analysis notebook for the synthetic process log built on Kaggle market rates.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/synthetic_deals.csv', parse_dates=['request_time','quote_time','client_accept_time','limit_check_time','trade_capture_time','confirmation_time','settlement_time'])
df.head()

In [ ]:
df['total_minutes'] = (df['settlement_time'] - df['request_time']).dt.total_seconds() / 60
summary = pd.Series({
    'deal_count': len(df),
    'avg_total_minutes': df['total_minutes'].mean(),
    'median_total_minutes': df['total_minutes'].median(),
    'sla_breach_rate': df['sla_breach'].mean(),
    'manual_processing_rate': df['manual_processing_flag'].mean(),
    'stp_rate': (~df['manual_processing_flag']).mean(),
    'first_pass_yield': ((df['deal_status'].eq('settled')) & (df['rework_count'].eq(0))).mean(),
})
summary

In [ ]:
stage_pairs = [('request_to_quote','request_time','quote_time'),('quote_to_accept','quote_time','client_accept_time'),('accept_to_limit','client_accept_time','limit_check_time'),('limit_to_capture','limit_check_time','trade_capture_time'),('capture_to_confirmation','trade_capture_time','confirmation_time'),('confirmation_to_settlement','confirmation_time','settlement_time')]
stages = pd.DataFrame([{'stage': n, 'avg_minutes': (df[e]-df[s]).dt.total_seconds().mean()/60} for n,s,e in stage_pairs])
stages.sort_values('avg_minutes').plot.barh(x='stage', y='avg_minutes', legend=False, title='Average minutes by stage')
plt.xlabel('minutes')

In [ ]:
df.groupby('exception_type').agg(deals=('deal_id','count'), sla_breach_rate=('sla_breach','mean'), avg_rework=('rework_count','mean')).sort_values('deals', ascending=False)